# **Tutorial 11 - Multi-Layer Networks: Width vs Depth**: Supplementary Interactive Material

-------------------------------------------------------------------------

Tutorial 10 fit $f$ with one hidden layer of 400 neurons. Here we spend a comparable budget differently — four hidden layers of 90 neurons — and train it on exactly the same data.

The setup cells below (target function, sampling, `DataLoader`, training loop) are repeated from Tutorial 10 so that this notebook runs on its own.


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [2]:
def f(x):
    return np.log(x) + (np.sin(x))**3 +(1/3)*np.cos(x-2)**3

In [3]:
x = np.linspace(1,6,400) #Sampling of f(x) [1,6], 400 points

In [4]:
X_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(1) # <- initially (n) (1,n) 
y_tensor = torch.tensor(f(x), dtype=torch.float32).unsqueeze(1)
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True) # creates batches of randomized pairs

In [7]:
class Function_Approximator_MLP(nn.Module):
    
        def __init__(self, hidden_size):
            super(Function_Approximator_MLP, self).__init__()
            self.fc1 = nn.Linear(1, hidden_size) #Four hidden layers 
            self.fc2 = nn.Linear(hidden_size, hidden_size)
            self.fc3 = nn.Linear(hidden_size, hidden_size)
            self.fc4 = nn.Linear(hidden_size, hidden_size)
            self.fc5 = nn.Linear(hidden_size,1)

        def forward(self, x):
            x = F.relu(self.fc1(x))
            x = F.relu(self.fc2(x))
            x = F.relu(self.fc3(x))
            x = F.relu(self.fc4(x))
            x = self.fc5(x)
            return x


In [8]:
def train_model_with_evaluation(model, training_data, criterion, optimizer, epochs=100, eval_interval=1):
    model.train()
    
    prediction_history = [] #Creates an array to store past predictions
    epoch_snapshots = []
    
    X_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(1)
    
    for epoch in range(epochs):
        running_loss = 0.0
        
        # Training phase
        model.train() # used to un-freeze the weights and biases 
        for data, target in training_data:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Evaluation phase - every eval_interval epochs
        if (epoch + 1) % eval_interval == 0:
            model.eval()
            with torch.no_grad():
                predictions = model(X_tensor).squeeze().numpy()
                prediction_history.append(predictions.copy())
                epoch_snapshots.append(epoch + 1)
            
            #print(f'Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(training_data):.4f}')
        
    print("Training complete!")
    return prediction_history, epoch_snapshots

In [21]:
model = Function_Approximator_MLP(90)
learning_rate = 0.004
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.MSELoss()

In [22]:
prediction_history, epoch_snapshots = train_model_with_evaluation(model,train_loader,criterion,optimizer)

Training complete!


In [23]:
fig, ax1 = plt.subplots(1, 1, figsize=(6, 6))
ax1.set_xlim(1, 6)
ax1.set_ylim(0, 3)
ax1.grid(True)
ax1.set_title("Model Training: Ground Truth vs Predictions, 4 Layer, 240 Total Nodes")
ax1.set_xlabel('x')
ax1.plot(x, f(x), 'b-', linewidth=2, label='Ground Truth', alpha=0.7)
pred_line, = ax1.plot([], [], 'r--', linewidth=2, label='Predictions')
epoch_text = ax1.text(0.02, 0.95, '', transform=ax1.transAxes, fontsize=12, 
                     bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
ax1.legend()
plt.tight_layout()

def update(frame):
    # Update prediction line
    current_predictions = prediction_history[frame]
    pred_line.set_data(x, current_predictions)
    
    # Update epoch display
    epoch_text.set_text(f'Epoch = {epoch_snapshots[frame]}')
    
    return pred_line, epoch_text

ani = FuncAnimation(fig, update, frames=len(prediction_history), interval=100, blit=True, repeat=True)
# Display
plt.close(fig)  
HTML(ani.to_jshtml())